In [ ]:
# @title Run
# @markdown If you are using a smartphone, please select the desktop site option.

!apt-get install libzbar0 > /dev/null 2>&1
!pip install qrcode pyzbar ipywidgets > /dev/null 2>&1

import os
import json
import random
import string
import io
import ipywidgets as widgets
import base64
from IPython.display import display, clear_output, HTML, Javascript
from google.colab import drive
import qrcode
from PIL import Image
from pyzbar.pyzbar import decode

css_style = """
<style>
    .widget-html, .widget-label, .widget-text-label, .widget-dropdown-label, .widget-textarea-label, h3, .output_subarea {
        color: #f5f5f5 !important;
    }
    .widget-text input, .widget-textarea textarea, select {
        background-color: #333333 !important;
        color: #ffffff !important;
        border: 1px solid #333333 !important;
        padding: 5px;
    }
    select option {
        background-color: #333333 !important;
        color: #ffffff !important;
    }
.p-TabBar {
    border-bottom: 1px solid #555555 !important;
}
.p-TabBar, .widget-tab-contents {
    background-color: transparent !important;
}
.p-TabBar-tab {
    background-color: #2A2A2A !important;
    border: 1px solid #444444 !important;
    border-bottom: none !important;
    border-radius: 4px 4px 0 0 !important;
    margin-right: 2px !important;
}

.p-TabBar-tabLabel {
    color: #bbbbbb !important;
    font-weight: bold;
}

.p-TabBar-tab:hover {
    background-color: #333333 !important;
}

.p-TabBar-tab.p-mod-current {
    background-color: #333333 !important;
    border-color: #42a5f5 !important;
    border-bottom: none !important;
}
.p-TabBar-tab.p-mod-current .p-TabBar-tabLabel {
    color: #42a5f5 !important;
}
    h3.ui-header {
        border-left: 5px solid #2196F3;
        padding-left: 10px;
        margin-top: 20px;
        margin-bottom: 15px;
        color: #90CAF9 !important;
        background-color: #263238;
        padding-top: 5px;
        padding-bottom: 5px;
        border-radius: 0 5px 5px 0;
    }
    .app-container {
        background-color: #212121;
        padding: 20px;
        border-radius: 10px;
        box-shadow: 0 4px 8px rgba(0,0,0,0.5);
    }
    .qr-image-wrapper {
        cursor: pointer;
    }
    .duplicate-item {
        border-radius: 3px;
    }
.widget-html, .widget-label, .widget-text-label, .widget-dropdown-label, .widget-textarea-label,
h1, h2, h3, h4, h5, h6,
.output_subarea {
    color: #f5f5f5 !important;
}
h2.ui-header, h3.ui-header {
}
.custom-accordion-button {
    box-sizing: border-box !important;
    background-color: #2A2A2A !important;
    border: 1px solid #444444 !important;

    padding: 10px 12px !important;
    text-align: left !important;
    cursor: pointer !important;
    color: #FF5252 !important;
    font-weight: bold !important;
    font-size: 1.1em !important;
    border-radius: 4px !important;
    line-height: 0.5 !important;
}
.custom-accordion-button .widget-button-text {
    color: #FF5252 !important;
    font-weight: bold !important;
    vertical-align: middle !important;
}
.custom-accordion-button.expanded {
    border-radius: 4px 4px 0 0 !important;
    background-color: #333333 !important;
}
.custom-accordion-button:hover {
    background-color: #333333 !important;
}

</style>
"""

APP_WIDTH = '800px'
LABEL_WIDTH = '80px'
LAYOUT_APP = widgets.Layout(width=APP_WIDTH, margin='0 auto')
LAYOUT_HALF = widgets.Layout(width='48%', margin='2px 0')
LAYOUT_FULL = widgets.Layout(width='98%', margin='2px 0')
LAYOUT_BTN = widgets.Layout(width='auto', margin='10px 0')
STYLE_COMMON = {'description_width': LABEL_WIDTH}

def make_list(name, count):
    return ["Not Specified"] + [f"{i}" for i in range(count)]

ANTENNA_LIST = ["Not Specified"] + [
    "No Antenna", "Sole Guard", "Heal", "Revive",
    "Antidote", "Mobilize", "Awaken", "Cure Element", "Cure Status","Mighty", "Excite",
    "Steroids",
    "Rage", "Wall", "Haste", "Dodge",
    "Poison", "Lullaby", "Paralyze", "Blind",
    "Weaken", "Insult",
    "Chain", "All Defocus", "Fireball", "Fire Cracker",
    "Icicle", "On the Rocks", "Whirlwind", "Breeze", "Falling Rock",
    "Pebbles", "Static", "Lightning", "Water Gun", "Bucket of Water",
    "Weak Flash", "Spotlight", "Scare", "Dark Ball", "Knock Down"
]

BODY_LIST = ["Not Specified"] + [
    "1A(0 Fastest)", "1B(0 Fast)", "1C(0 Average)", "1D(0 Large)", "1E(0 Largest)",
    "2A(3+ Fastest)", "2B(3+ Fast)", "2C(3+ Average)", "2D(3+ Large)", "2E(3+ Largest)",
    "3A(3- Fastest)", "3B(3- Fast)", "3C(3- Average)", "3D(3- Large)", "3E(3- Largest)",
    "4A(6+ Fastest)", "4B(6+ Fast)", "4C(6+ Average)", "4D(6+ Large)", "4E(6+ Largest)",
    "5A(10 Fastest)", "5B(6- Fast)", "5C(6- Average)", "5D(6- Large)", "5E(6- Largest)",
    "6A(15 Fastest)", "6B(15 Fast)", "6C(15 Average)", "6D(10 Large)", "6E(10 Largest)"
]

COLOR_LIST = ["Not Specified"] + [
    "Black", "Darker Black", "Lighter Black", "Red", "Darker Red", "Lighter Red",
    "Cyan", "Darker Cyan", "Lighter Cyan", "Green", "Darker Green", "Lighter Green", "Orange", "Darker Orange",
    "Lighter Orange", "Yellow", "Darker Yellow", "Lighter Yellow", "Blue", "Darker Blue", "Lighter Blue", "White",
    "Darker White", "Lighter White", "Purple", "Darker Purple", "Lighter Purple", "Pink", "Darker Pink", "Lighter Pink",
    "Gold", "Darker Gold", "Lighter Gold", "Silver", "Darker Silver", "Lighter Silver"
]

HAIR_SHAPE = make_list("Face Type", 96)
HAIR_COLOR = make_list("Hair Color", 27)
SKIN_COLOR = make_list("Skin Tone", 17)
EYE_LIST = make_list("Eyes", 69)
MOUTH_LIST = make_list("Mouth", 72)
NOSE_LIST = make_list("Nose", 43)
EYEBROW_LIST = make_list("Eyebrows", 40)
CHEEK_LIST = make_list("Cheeks", 15)

try:
    drive.mount('/content/drive', force_remount=True)
except:
    pass

BASE_DIR = '/content/drive/MyDrive/denpaxqr'
DB_FILE = os.path.join(BASE_DIR, 'database.json')

BACKUP_DIR = os.path.join(BASE_DIR, 'backups')
if not os.path.exists(BACKUP_DIR):
    os.makedirs(BACKUP_DIR)

def load_db():
    if not os.path.exists(DB_FILE): return []
    try:
        with open(DB_FILE, 'r', encoding='utf-8') as f: return json.load(f)
    except: return []

def save_db(data):
    with open(DB_FILE, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=4, ensure_ascii=False)

def generate_random_str(length):
    chars = string.ascii_letters + string.digits
    return ''.join(random.choice(chars) for _ in range(length))

GSHEET_FACE_PARTS_URL = "https://docs.google.com/spreadsheets/d/e/2PACX-1vRv_PQpeUfZOY3lMqrKfkH873bazG55xhtQkjV2gEYrVxT7UzfmmY9fSj41W4IjavXiOueNXjdgU7tk/pubhtml"

gsheet_content_iframe = f"""
<div style='color: #333333;'>
    <iframe src="{GSHEET_FACE_PARTS_URL}" width="100%" height="400" frameborder="0"></iframe>
    <p style="color:#ffffff; font-size:0.9em;">※This table was used in Scylla.</p>
</div>
"""

def create_qr_widget(text, size=200):
    qr = qrcode.QRCode(
        version=None,
        box_size=5,
        border=2
    )
    qr.add_data(text)
    qr.make(fit=True)
    pil_img = qr.make_image(fill_color="black", back_color="white")

    img_byte_arr = io.BytesIO()
    pil_img.save(img_byte_arr, format='PNG')
    img_base64 = base64.b64encode(img_byte_arr.getvalue()).decode('utf-8')

    js_func = f"""
    var qrData = 'data:image/png;base64,{img_base64}';
    var newWindow = window.open("", "_blank", "width=400,height=400");
    newWindow.document.write('<html><body style="margin:0; text-align:center;"><img src="' + qrData + '" style="width:100%; height:100%; image-rendering: pixelated; margin:0;"></body></html>');
    newWindow.document.title = 'QR Code Zoom';
    """

    html_img = f"""
    <div onclick=" {js_func.replace('"', '&quot;')} " class='qr-image-wrapper'>
        <img src="data:image/png;base64,{img_base64}" width="{size}" height="{size}" style="image-rendering: pixelated;">
    </div>
    """

    return widgets.HTML(html_img, layout=widgets.Layout(width=f'{size}px', height=f'{size}px'))

qr_source_text = widgets.Text(description='QR data:', placeholder='Enter text', style=STYLE_COMMON, layout=LAYOUT_FULL)
rand_len = widgets.IntSlider(value=13, min=1, max=126, description='Count:', style=STYLE_COMMON, layout=widgets.Layout(width='40%'))
btn_gen_manual = widgets.Button(description='Input text', button_style='info', icon='qrcode', layout=LAYOUT_BTN)
btn_gen_random = widgets.Button(description='Random', button_style='warning', icon='random', layout=LAYOUT_BTN)
qr_output = widgets.Output()

input_name = widgets.Text(description='Name:', style=STYLE_COMMON, layout=LAYOUT_HALF)
input_color = widgets.Dropdown(options=COLOR_LIST, description='Color:', style=STYLE_COMMON, layout=LAYOUT_HALF)
input_antenna = widgets.Dropdown(options=ANTENNA_LIST, description='Antenna:', style=STYLE_COMMON, layout=LAYOUT_HALF)
input_body = widgets.Dropdown(options=BODY_LIST, description='Body Type:', style=STYLE_COMMON, layout=LAYOUT_HALF)

hair_shape = widgets.Dropdown(options=HAIR_SHAPE, description='Face Type:', style=STYLE_COMMON, layout=LAYOUT_HALF)
hair_color = widgets.Dropdown(options=HAIR_COLOR, description='Hair Color:', style=STYLE_COMMON, layout=LAYOUT_HALF)
skin_color = widgets.Dropdown(options=SKIN_COLOR, description='Skin Tone:', style=STYLE_COMMON, layout=LAYOUT_HALF)
eye_part = widgets.Dropdown(options=EYE_LIST, description='Eyes:', style=STYLE_COMMON, layout=LAYOUT_HALF)
mouth_part = widgets.Dropdown(options=MOUTH_LIST, description='Mouth:', style=STYLE_COMMON, layout=LAYOUT_HALF)
nose_part = widgets.Dropdown(options=NOSE_LIST, description='Nose:', style=STYLE_COMMON, layout=LAYOUT_HALF)
eyebrow_part = widgets.Dropdown(options=EYEBROW_LIST, description='Eyebrows:', style=STYLE_COMMON, layout=LAYOUT_HALF)
cheek_part = widgets.Dropdown(options=CHEEK_LIST, description='Cheeks:', style=STYLE_COMMON, layout=LAYOUT_HALF)

input_memo = widgets.Textarea(description='Memo:', placeholder='You can write notes here.', style=STYLE_COMMON, layout=LAYOUT_FULL)

btn_register = widgets.Button(description='Register with this content', button_style='success', icon='save', layout=widgets.Layout(width='98%', height='40px'))
msg_output = widgets.Output()

current_qr_data = {"text": ""}

input_fields = {
    'name': input_name, 'color': input_color, 'antenna': input_antenna, 'body': input_body,
    'hair_shape': hair_shape, 'hair_color': hair_color, 'skin_color': skin_color,
    'eye': eye_part, 'mouth': mouth_part, 'nose': nose_part, 'eyebrow': eyebrow_part,
    'cheek': cheek_part, 'memo': input_memo
}

def reset_registration_fields():
    qr_source_text.value = ''
    current_qr_data["text"] = ''
    for key, widget in input_fields.items():
        if isinstance(widget, widgets.Text) or isinstance(widget, widgets.Textarea):
            widget.value = ''
        elif isinstance(widget, widgets.Dropdown):
            widget.value = widget.options[0]
    with qr_output:
        clear_output()

def on_click_gen_manual(b):
    with qr_output:
        clear_output()
        if not qr_source_text.value:
            print("Please enter text.")
            return
        current_qr_data["text"] = qr_source_text.value
        display(create_qr_widget(qr_source_text.value))
        print(f"Data: {qr_source_text.value}")

def on_click_gen_random(b):
    with qr_output:
        clear_output()
        txt = generate_random_str(rand_len.value)
        qr_source_text.value = txt
        current_qr_data["text"] = txt
        display(create_qr_widget(txt))
        print(f"Data: {txt}")

btn_gen_manual.on_click(on_click_gen_manual)
btn_gen_random.on_click(on_click_gen_random)

def on_click_register(b):
    with msg_output:
        clear_output()
        if not current_qr_data["text"]:
            print("Generate the QR code for us first.")
            return
        if not input_name.value:
            print("Please enter name.")
            return

        new_record = {
            "qr_data": current_qr_data["text"],
            "name": input_name.value,
            "antenna": input_antenna.value,
            "color": input_color.value,
            "body": input_body.value,
            "memo": input_memo.value,
            "hair_shape": hair_shape.value, "hair_color": hair_color.value,
            "skin_color": skin_color.value, "eye": eye_part.value,
            "mouth": mouth_part.value, "nose": nose_part.value,
            "eyebrow": eyebrow_part.value, "cheek": cheek_part.value
        }

        db = load_db()
        db.append(new_record)
        save_db(db)
        print(f"Done!: {input_name.value}.You can register next.")

        reset_registration_fields()

btn_register.on_click(on_click_register)

def create_custom_accordion(title_text, content_widget, initial_open=False):

    content_container = widgets.VBox(
        [content_widget],
        layout=widgets.Layout(
            display=('block' if initial_open else 'none'),
            width='100%',
            border='1px solid #444',
            border_top='none',
            padding='10px',
            background_color='#212121',
            border_radius='0 0 4px 4px'
        )
    )
    icon = '▼' if initial_open else '▶'
    header_text = f"{icon} {title_text}"

    header_button = widgets.Button(
        description=header_text,
        layout=widgets.Layout(width='100%', margin='5px 0 0 0'),
        style=widgets.ButtonStyle(font_weight='bold')
    )
    header_button.add_class('custom-accordion-button')

    header_button.add_class('expanded' if initial_open else 'closed')

    is_open = initial_open

    def on_toggle_click(b):
        nonlocal is_open
        is_open = not is_open

        if is_open:
            content_container.layout.display = 'block'
            b.description = f"▼ {title_text}"
            b.remove_class('closed')
            b.add_class('expanded')
        else:
            content_container.layout.display = 'none'
            b.description = f"▶ {title_text}"
            b.remove_class('expanded')
            b.add_class('closed')

    header_button.on_click(on_toggle_click)

    return widgets.VBox([
        header_button,
        content_container
    ], layout=LAYOUT_FULL)

gsheet_html_tab1 = widgets.HTML(value=gsheet_content_iframe, layout=LAYOUT_FULL)

accordion_tab1 = create_custom_accordion('Face Parts Chart (Click to view)', gsheet_html_tab1, initial_open=False)
accordion_tab1.selected_index = None

tab1_ui = widgets.VBox([
    widgets.HTML("<h3 class='ui-header'>1. Generate QR Code</h3>"),
    qr_source_text,
    widgets.HBox([rand_len, btn_gen_random, btn_gen_manual]),
    qr_output,
    widgets.HTML("<h3 class='ui-header'>2. Register</h3>"),
    widgets.HBox([input_name, input_color]),
    widgets.HBox([input_antenna, input_body]),

    widgets.HTML("<h3 class='ui-header'>Face parts</h3>"),
    widgets.HBox([hair_shape, hair_color]),
    widgets.HBox([skin_color, eye_part]),
    widgets.HBox([mouth_part, nose_part]),
    widgets.HBox([eyebrow_part, cheek_part]),

    accordion_tab1,

    input_memo,
    btn_register,
    msg_output
])

search_list_antenna = ANTENNA_LIST
search_list_color = COLOR_LIST
search_list_body = BODY_LIST

search_name = widgets.Text(description='Name:', style=STYLE_COMMON, layout=LAYOUT_HALF)
search_color = widgets.Dropdown(options=search_list_color, description='Color:', style=STYLE_COMMON, layout=LAYOUT_HALF)
search_antenna = widgets.Dropdown(options=search_list_antenna, description='Antenna:', style=STYLE_COMMON, layout=LAYOUT_HALF)
search_body = widgets.Dropdown(options=search_list_body, description='Body Type:', style=STYLE_COMMON, layout=LAYOUT_HALF)

search_hair_shape = widgets.Dropdown(options=HAIR_SHAPE, description='Face Type:', style=STYLE_COMMON, layout=LAYOUT_HALF)
search_hair_color = widgets.Dropdown(options=HAIR_COLOR, description='Hair Color:', style=STYLE_COMMON, layout=LAYOUT_HALF)
search_skin_color = widgets.Dropdown(options=SKIN_COLOR, description='Skin Tone:', style=STYLE_COMMON, layout=LAYOUT_HALF)
search_eye_part = widgets.Dropdown(options=EYE_LIST, description='Eyes:', style=STYLE_COMMON, layout=LAYOUT_HALF)
search_mouth_part = widgets.Dropdown(options=MOUTH_LIST, description='Mouth:', style=STYLE_COMMON, layout=LAYOUT_HALF)
search_nose_part = widgets.Dropdown(options=NOSE_LIST, description='Nose:', style=STYLE_COMMON, layout=LAYOUT_HALF)
search_eyebrow_part = widgets.Dropdown(options=EYEBROW_LIST, description='Eyebrows:', style=STYLE_COMMON, layout=LAYOUT_HALF)
search_cheek_part = widgets.Dropdown(options=CHEEK_LIST, description='Cheeks:', style=STYLE_COMMON, layout=LAYOUT_HALF)

btn_search = widgets.Button(description='Search', button_style='primary', icon='search', layout=LAYOUT_BTN)
search_output = widgets.Output()

def match_field(condition, value):
    return (condition == "Not Specified") or (condition == value)

def on_click_search(b):
    with search_output:
        clear_output()
        db = load_db()
        results = []

        for record in db:

            if search_name.value and search_name.value not in record.get('name', ''): continue
            if not match_field(search_color.value, record.get('color')): continue
            if not match_field(search_antenna.value, record.get('antenna')): continue
            if not match_field(search_body.value, record.get('body')): continue
            if not match_field(search_hair_shape.value, record.get('hair_shape')): continue
            if not match_field(search_hair_color.value, record.get('hair_color')): continue
            if not match_field(search_skin_color.value, record.get('skin_color')): continue
            if not match_field(search_eye_part.value, record.get('eye')): continue
            if not match_field(search_mouth_part.value, record.get('mouth')): continue
            if not match_field(search_nose_part.value, record.get('nose')): continue
            if not match_field(search_eyebrow_part.value, record.get('eyebrow')): continue
            if not match_field(search_cheek_part.value, record.get('cheek')): continue

            results.append(record)

        if not results:
            print("Not found.")
            return

        print(f"{len(results)} hits found.")

        all_cards = []

        for r in results:
            face_parts_info = (
                f"Face Type: {r.get('hair_shape', 'N/A')}, Hair Color: {r.get('hair_color', 'N/A')}, Skin Tone: {r.get('skin_color', 'N/A')}<br>"
                f"Eyes: {r.get('eye', 'N/A')}, Mouth: {r.get('mouth', 'N/A')}, Nose: {r.get('nose', 'N/A')}, Eyebrows: {r.get('eyebrow', 'N/A')}, Cheeks: {r.get('cheek', 'N/A')}"
            )

            info_text_html = f"""
                <div style='flex-grow: 1; padding-right: 10px;'>
                    <div style='font-size: 1.2em; font-weight: bold; color: #4fc3f7;'>{r.get('name', 'No Name')}</div>
                    <div style='color: #fff;'>Antenna: {r.get('antenna', 'N/A')} / Color: {r.get('color', 'N/A')} / Body Type: {r.get('body', 'N/A')}</div>
                    <div style='color: #bbb; font-size: 0.9em; margin-top: 5px;'>**Face parts:**<br>{face_parts_info}</div>
                    <div style='color: #bbb; font-size: 0.9em; margin-top: 5px;'>**Memo:** {r.get('memo', '')}</div>
                </div>
            """

            qr_widget = create_qr_widget(r.get('qr_data', ''), size=150)

            combined_content = widgets.HBox([widgets.HTML(info_text_html), qr_widget],
                                            layout=widgets.Layout(align_items='center', width='100%'))

            final_card = widgets.VBox([combined_content],
                                          layout=widgets.Layout(border='1px solid #444',
                                                                background_color='#2c2c2c',
                                                                padding='10px',
                                                                margin='0 0 10px 0',
                                                                border_radius='5px'))

            all_cards.append(final_card)

        display(widgets.VBox(all_cards))


btn_search.on_click(on_click_search)

gsheet_html_tab2 = widgets.HTML(value=gsheet_content_iframe, layout=LAYOUT_FULL)

accordion_tab2 = create_custom_accordion('Face Parts Chart (Click to view)', gsheet_html_tab1, initial_open=False)
accordion_tab2.selected_index = None


tab2_ui = widgets.VBox([
    widgets.HTML("<h3 class='ui-header'>Basic Information</h3>"),
    widgets.HBox([search_name, search_color]),
    widgets.HBox([search_antenna, search_body]),

    widgets.HTML("<h3 class='ui-header'>Face parts</h3>"),
    widgets.HBox([search_hair_shape, search_hair_color]),
    widgets.HBox([search_skin_color, search_eye_part]),
    widgets.HBox([search_mouth_part, search_nose_part]),
    widgets.HBox([search_eyebrow_part, search_cheek_part]),

    accordion_tab2,

    btn_search,
    search_output
])

edit_search_name = widgets.Text(description='Name:', placeholder='Enter part of the name', style=STYLE_COMMON, layout=widgets.Layout(width='60%'))
btn_edit_search = widgets.Button(description='Search', button_style='info', icon='search', layout=widgets.Layout(width='auto'))
edit_select_list = widgets.Select(options=[], description='Select:', style=STYLE_COMMON,
                                     layout=widgets.Layout(width='98%', height='100px'))
edit_output_msg = widgets.Output()

edit_name = widgets.Text(description='Name:', style=STYLE_COMMON, layout=LAYOUT_HALF, disabled=True)
edit_color = widgets.Dropdown(options=COLOR_LIST, description='Color:', style=STYLE_COMMON, layout=LAYOUT_HALF, disabled=True)
edit_antenna = widgets.Dropdown(options=ANTENNA_LIST, description='Antenna:', style=STYLE_COMMON, layout=LAYOUT_HALF, disabled=True)
edit_body = widgets.Dropdown(options=BODY_LIST, description='Body Type:', style=STYLE_COMMON, layout=LAYOUT_HALF, disabled=True)
edit_qr_data = widgets.Textarea(description='QR Data:', style=STYLE_COMMON, layout=LAYOUT_FULL, disabled=True)
edit_memo = widgets.Textarea(description='Memo:', style=STYLE_COMMON, layout=LAYOUT_FULL, disabled=True)
edit_qr_widget = widgets.Output()

edit_hair_shape = widgets.Dropdown(options=HAIR_SHAPE, description='Face Type:', style=STYLE_COMMON, layout=LAYOUT_HALF, disabled=True)
edit_hair_color = widgets.Dropdown(options=HAIR_COLOR, description='Hair Color:', style=STYLE_COMMON, layout=LAYOUT_HALF, disabled=True)
edit_skin_color = widgets.Dropdown(options=SKIN_COLOR, description='Skin Tone:', style=STYLE_COMMON, layout=LAYOUT_HALF, disabled=True)
edit_eye_part = widgets.Dropdown(options=EYE_LIST, description='Eyes:', style=STYLE_COMMON, layout=LAYOUT_HALF, disabled=True)
edit_mouth_part = widgets.Dropdown(options=MOUTH_LIST, description='Mouth:', style=STYLE_COMMON, layout=LAYOUT_HALF, disabled=True)
edit_nose_part = widgets.Dropdown(options=NOSE_LIST, description='Nose:', style=STYLE_COMMON, layout=LAYOUT_HALF, disabled=True)
edit_eyebrow_part = widgets.Dropdown(options=EYEBROW_LIST, description='Eyebrows:', style=STYLE_COMMON, layout=LAYOUT_HALF, disabled=True)
edit_cheek_part = widgets.Dropdown(options=CHEEK_LIST, description='Cheeks:', style=STYLE_COMMON, layout=LAYOUT_HALF, disabled=True)

edit_ui_parts = [edit_name, edit_color, edit_antenna, edit_body, edit_qr_data, edit_memo,
                 edit_hair_shape, edit_hair_color, edit_skin_color, edit_eye_part, edit_mouth_part,
                 edit_nose_part, edit_eyebrow_part, edit_cheek_part]

btn_update_exec = widgets.Button(description='Save Edits', button_style='success', icon='save', disabled=True, layout=widgets.Layout(width='48%'))
btn_delete_exec = widgets.Button(description='Delete Execution', button_style='danger', icon='trash', disabled=True, layout=widgets.Layout(width='48%'))

current_edit_idx = None

def toggle_edit_ui(disabled):
    for widget in edit_ui_parts:
        widget.disabled = disabled
    btn_update_exec.disabled = disabled
    btn_delete_exec.disabled = disabled

def fill_edit_ui(record):
    edit_name.value = record.get('name', '')
    edit_color.value = record.get('color', 'Not Specified')
    edit_antenna.value = record.get('antenna', 'Not Specified')
    edit_body.value = record.get('body', 'Not Specified')
    edit_qr_data.value = record.get('qr_data', '')
    edit_memo.value = record.get('memo', '')

    edit_hair_shape.value = record.get('hair_shape', 'Not Specified')
    edit_hair_color.value = record.get('hair_color', 'Not Specified')
    edit_skin_color.value = record.get('skin_color', 'Not Specified')
    edit_eye_part.value = record.get('eye', 'Not Specified')
    edit_mouth_part.value = record.get('mouth', 'Not Specified')
    edit_nose_part.value = record.get('nose', 'Not Specified')
    edit_eyebrow_part.value = record.get('eyebrow', 'Not Specified')
    edit_cheek_part.value = record.get('cheek', 'Not Specified')

    with edit_qr_widget:
        clear_output()
        display(create_qr_widget(record.get('qr_data', '')))

def on_click_edit_search(b):
    global current_edit_idx
    with edit_output_msg:
        clear_output()
        db = load_db()
        display_options = []
        target = edit_search_name.value
        current_edit_idx = None
        toggle_edit_ui(True)

        edit_select_list.value = None

        for widget in edit_ui_parts:
            if isinstance(widget, widgets.Text) or isinstance(widget, widgets.Textarea):
                widget.value = ''
            elif isinstance(widget, widgets.Dropdown):
                widget.value = widget.options[0]
        with edit_qr_widget: clear_output()


        for i, r in enumerate(db):
            if not target or (target in r.get('name', '')):
                label = f"No.{i} | {r.get('name', 'No Name')} ({r.get('antenna', 'N/A')}, {r.get('color', 'N/A')})"
                display_options.append((label, i))

        if not display_options:
            edit_select_list.options = []
            print("Not applicable")
        else:
            edit_select_list.options = display_options
            print(f"{len(display_options)} items found. Please select from the list.")

def on_select_change(change):
    global current_edit_idx
    with edit_output_msg:
        clear_output()
        idx = change['new']
        if idx is not None:
            current_edit_idx = idx
            db = load_db()
            record = db[idx]
            fill_edit_ui(record)
            toggle_edit_ui(False)
            print(f"Editing Target: No.{idx} {record.get('name', 'No Name')}")
        else:
            current_edit_idx = None
            toggle_edit_ui(True)
            with edit_qr_widget: clear_output()
            print("Please select Denpa Men from the list.")

edit_select_list.observe(on_select_change, names='value')

def on_click_update_exec(b):
    global current_edit_idx
    with edit_output_msg:
        clear_output()
        if current_edit_idx is None:
            print("No editing target is selected.")
            return
        if not edit_name.value:
            print("Name is required.")
            return
        if not edit_qr_data.value:
            print("QR data is required.")
            return

        db = load_db()
        db[current_edit_idx] = {
            "qr_data": edit_qr_data.value,
            "name": edit_name.value,
            "antenna": edit_antenna.value,
            "color": edit_color.value,
            "body": edit_body.value,
            "memo": edit_memo.value,
            "hair_shape": edit_hair_shape.value, "hair_color": edit_hair_color.value,
            "skin_color": edit_skin_color.value, "eye": edit_eye_part.value,
            "mouth": edit_mouth_part.value, "nose": edit_nose_part.value,
            "eyebrow": edit_eyebrow_part.value, "cheek": edit_cheek_part.value
        }
        save_db(db)
        print(f"Editing complete: {edit_name.value} (No.{current_edit_idx}).The list has been updated.")

        edit_select_list.value = None
        on_click_edit_search(None)
        toggle_edit_ui(True)

btn_update_exec.on_click(on_click_update_exec)

def on_click_delete_exec(b):
    global current_edit_idx
    with edit_output_msg:
        clear_output()
        if current_edit_idx is None:
            print("No items have been selected for deletion.")
            return

        db = load_db()
        rm = db.pop(current_edit_idx)
        save_db(db)
        print(f"Deleted: {rm.get('name', 'No Name')}.The list has been updated.")

        current_edit_idx = None
        toggle_edit_ui(True)
        for widget in edit_ui_parts:
            if isinstance(widget, widgets.Text) or isinstance(widget, widgets.Textarea):
                widget.value = ''
            elif isinstance(widget, widgets.Dropdown):
                widget.value = widget.options[0]
        with edit_qr_widget: clear_output()

        on_click_edit_search(None)

btn_delete_exec.on_click(on_click_delete_exec)
btn_edit_search.on_click(on_click_edit_search)

gsheet_html_tab3 = widgets.HTML(value=gsheet_content_iframe, layout=LAYOUT_FULL)

accordion_tab3 = create_custom_accordion('Face Parts Chart (Click to view)', gsheet_html_tab1, initial_open=False)
accordion_tab3.selected_index = None

tab3_ui = widgets.VBox([
    widgets.HTML("<h3 class='ui-header'>Data Search and Selection</h3>"),
    widgets.HBox([edit_search_name, btn_edit_search]),
    edit_output_msg,
    edit_select_list,
    widgets.HTML("<h3 class='ui-header'>Edit Area</h3>"),
    widgets.HBox([edit_name, edit_color]),
    widgets.HBox([edit_antenna, edit_body]),

    widgets.HTML("<h3 class='ui-header'>Face parts</h3>"),
    widgets.HBox([edit_hair_shape, edit_hair_color]),
    widgets.HBox([edit_skin_color, edit_eye_part]),
    widgets.HBox([edit_mouth_part, edit_nose_part]),
    widgets.HBox([edit_eyebrow_part, edit_cheek_part]),

    accordion_tab3,

    edit_qr_data,
    edit_memo,
    edit_qr_widget,
    widgets.HBox([btn_update_exec, btn_delete_exec])
])

upload_qr = widgets.FileUpload(
    accept='.png,.jpg,.jpeg',
    multiple=False,
    description="Upload QR code",
    layout=widgets.Layout(width='200px')
)
decode_btn = widgets.Button(description="Execute", button_style='primary', icon='barcode')
decode_output = widgets.Output()

def decode_qr_image(b):
    with decode_output:
        clear_output()

        if not upload_qr.value:
            print("Please upload the QR code.")
            return

        file = list(upload_qr.value.values())[0]
        img_bytes = file['content']
        img = Image.open(io.BytesIO(img_bytes))

        decoded = decode(img)

        if not decoded:
            print("Unable to scan QR code.")
            return

        qr_text = decoded[0].data.decode('utf-8')
        qr_source_text.value = qr_text
        current_qr_data["text"] = qr_text

        print("Decoding successful! Updated on the registration screen.")
        with qr_output:
            clear_output()
            display(create_qr_widget(qr_text))
            print(f"Data: {qr_text}")

        display(create_qr_widget(qr_text, size=100))
        print("The QR data has been reflected in the “QR Data” field under the “Gen & Reg” tab.")

decode_btn.on_click(decode_qr_image)

tab4_ui = widgets.VBox([
    widgets.HTML("<h3 class='ui-header'>External QR Registration Mode</h3>"),
    upload_qr,
    decode_btn,
    decode_output
])

check_btn = widgets.Button(description='Data Duplication Check Execution', button_style='danger', icon='exclamation-triangle', layout=LAYOUT_FULL)
check_output = widgets.Output()

def delete_record_by_index(idx_to_delete):
    db = load_db()

    if 0 <= idx_to_delete < len(db):
        db.pop(idx_to_delete)
        save_db(db)

        on_click_check(None)
    else:
        with check_output:
            print(f"Error: Index {idx_to_delete} is invalid.")


def create_duplicate_row(idx, name, qr_data_preview, is_qr_dup=False):
    if is_qr_dup:
        info_text = f"No.{idx} / Name: {name}"
        color = '#ff9800'
    else: # Name Dup
        info_text = f"No.{idx} / QR data (first 10 characters): {qr_data_preview[:10]}..."
        color = '#ffeb3b'

    info_widget = widgets.HTML(info_text, layout=widgets.Layout(width='65%'))

    delete_button = widgets.Button(
        description='Delete this record',
        button_style='danger',
        icon='trash',
        layout=widgets.Layout(width='30%', margin='0 0 0 10px')
    )

    delete_button.on_click(lambda b, index=idx: delete_record_by_index(index))

    return widgets.HBox([info_widget, delete_button],
                        layout=widgets.Layout(align_items='center',
                                              border=f'1px solid {color}',
                                              padding='5px',
                                              margin='2px 0',
                                              background_color='#333333'))

def on_click_check(b):
    with check_output:
        clear_output()
        db = load_db()

        qr_duplicates = {}
        name_duplicates = {}

        for i, record in enumerate(db):
            qr_data = record.get('qr_data')
            name = record.get('name')

            if qr_data:
                if qr_data not in qr_duplicates:
                    qr_duplicates[qr_data] = []
                qr_duplicates[qr_data].append((i, name))

            if name:
                if name not in name_duplicates:
                    name_duplicates[name] = []
                name_duplicates[name].append((i, qr_data))

        report_cards = []

        qr_dups_found = False
        qr_sections = []
        for qr, items in qr_duplicates.items():
            if len(items) > 1:
                qr_dups_found = True

                header_text = f"Duplicate QR Data: `{qr}` (Total: {len(items)} items)"

                record_rows = []
                for idx, name in items:
                    record_rows.append(create_duplicate_row(idx, name, qr, is_qr_dup=True))

                qr_sections.append(widgets.VBox([widgets.HTML(header_text)] + record_rows, layout=LAYOUT_FULL))

        if qr_dups_found:
            qr_card = widgets.VBox([
                widgets.HTML("<h4>Duplicate QR Data (records with the same QR data)</h4><hr>", layout=LAYOUT_FULL)]
                + qr_sections
                , layout=widgets.Layout(border='2px solid #ff9800', padding='10px', background_color='#2c2c2c', margin='0 0 10px 0'))
            report_cards.append(qr_card)

        name_dups_found = False
        name_sections = []
        for name, items in name_duplicates.items():
            if len(items) > 1:
                name_dups_found = True

                header_text = f"Name Duplication:`{name}` (Total {len(items)} people)"

                record_rows = []
                for idx, qr_data_preview in items:
                    record_rows.append(create_duplicate_row(idx, name, qr_data_preview, is_qr_dup=False))

                name_sections.append(widgets.VBox([widgets.HTML(header_text)] + record_rows, layout=LAYOUT_FULL))

        if name_dups_found:
            name_card = widgets.VBox([
                widgets.HTML("<h4>Name Duplication (Records with the same name)</h4><hr>", layout=LAYOUT_FULL)]
                + name_sections
                , layout=widgets.Layout(border='2px solid #ffeb3b', padding='10px', background_color='#2c2c2c', margin='0 0 10px 0'))
            report_cards.append(name_card)

        if qr_dups_found or name_dups_found:
            print(f"A total of {len(report_cards)} duplicate entries were found.")
            display(widgets.VBox(report_cards))
            print("No. is the index number in the database registration order. Pressing the Delete button will immediately delete the data and update the list.")
        else:
            print("No duplicate data was found. The data is clean.")

check_btn.on_click(on_click_check)

tab5_ui = widgets.VBox([
    widgets.HTML("<h3 class='ui-header'>Duplicate Data Check</h3>"),
    widgets.HTML("""
        <p>Check for duplicates based on exact matches of “QR Data” and “Name,” and place a delete button on problematic records.<br>
        <strong>Warning: Deletion is performed immediately and cannot be undone.</strong>
        </p>
    """),
    check_btn,
    check_output
])


backup_header = widgets.HTML("<h4>Backup</h4>", layout=LAYOUT_FULL)
btn_backup = widgets.Button(description='Back up the database', button_style='info', icon='archive', layout=LAYOUT_FULL)
backup_msg_output = widgets.Output()

def create_backup_filename():
    from datetime import datetime
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    return f"database_backup_{timestamp}.json"

def on_click_backup(b):
    with backup_msg_output:
        clear_output()
        db = load_db()
        if not db:
            print("The database is empty, so backup cannot be performed.")
            return

        backup_filename = create_backup_filename()
        backup_path = os.path.join(BACKUP_DIR, backup_filename)

        try:
            with open(backup_path, 'w', encoding='utf-8') as f:
                json.dump(db, f, indent=4, ensure_ascii=False)
            print(f"Backup complete: {backup_filename}")
            update_restore_dropdown(None)
        except Exception as e:
            print(f"An error occurred during backup.: {e}")

btn_backup.on_click(on_click_backup)


restore_header = widgets.HTML("<h4>Restore</h4>", layout=LAYOUT_FULL)
warning_restore = widgets.HTML("<p style='color:#ff5722; font-weight:bold;'>Warning: Restoring will overwrite your current data.</p>", layout=LAYOUT_FULL)
dropdown_restore = widgets.Dropdown(options=['No backup'], description='Restore:', layout=widgets.Layout(width='60%'))
btn_restore = widgets.Button(description='Restore the selected file', button_style='danger', icon='sync', layout=widgets.Layout(width='30%'))
btn_refresh = widgets.Button(description='Update', icon='refresh', layout=widgets.Layout(width='auto'))

def update_restore_dropdown(b):
    try:
        files = [f for f in os.listdir(BACKUP_DIR) if f.startswith('database_backup_') and f.endswith('.json')]
        files.sort(reverse=True)
        if not files:
            dropdown_restore.options = ['No backup']
            btn_restore.disabled = True
        else:
            dropdown_restore.options = files
            btn_restore.disabled = False
    except FileNotFoundError:
        dropdown_restore.options = ['No backup']
        btn_restore.disabled = True
    except Exception as e:
        dropdown_restore.options = [f'Error: {e}']
        btn_restore.disabled = True

def on_click_restore(b):
    with backup_msg_output:
        clear_output()
        selected_file = dropdown_restore.value

        if selected_file == 'No backup' or 'Error' in selected_file:
            print("Select the file to restore.")
            return

        restore_path = os.path.join(BACKUP_DIR, selected_file)

        try:
            with open(restore_path, 'r', encoding='utf-8') as f:
                restored_data = json.load(f)

            save_db(restored_data)
            print(f"Completed: The data has been overwritten with {selected_file}'s data.")

        except Exception as e:
            print(f"An error occurred during restoration.: {e}")

btn_restore.on_click(on_click_restore)
btn_refresh.on_click(update_restore_dropdown)
update_restore_dropdown(None)

layout_common = widgets.Layout(width=APP_WIDTH, margin='20px auto', padding='20px',
                               background_color='#212121', border='1px solid #444', border_radius='10px')

backup_ui = widgets.VBox([
    backup_header,
    widgets.HTML("<p style='color:#ccc; margin-top:-10px;'>Save the current `database.json` file with a timestamp to the `backups/` folder.</p>"),
    btn_backup,
    backup_msg_output
])

restore_selection = widgets.HBox([dropdown_restore, btn_refresh], layout=widgets.Layout(align_items='flex-end'))
restore_ui = widgets.VBox([
    restore_header,
    warning_restore,
    restore_selection,
    btn_restore
])

tab6_ui = widgets.VBox([
    widgets.HTML("<h2 class='ui-header'>Data Management (Backup/Restore)</h2>"),
    backup_ui,
    restore_ui
])

tabs = widgets.Tab(children=[tab1_ui, tab2_ui, tab3_ui, tab4_ui, tab5_ui, tab6_ui])
tabs.set_title(0, 'Gen & Reg')
tabs.set_title(1, 'Search')
tabs.set_title(2, 'Edit')
tabs.set_title(3, 'External QR')
tabs.set_title(4, 'Data Check')
tabs.set_title(5, 'Backup')

app_container = widgets.VBox([tabs], layout=LAYOUT_APP)
app_container.add_class("app-container")

clear_output()
display(HTML(css_style))
display(app_container)



This is the English version. I don't usually use English, so there might be some incorrect expressions. Please be aware of this in advance.

Please strictly adhere to the following regarding handling.
1.   Prohibition of commercial use (including YouTube)
2.   Prohibition of Redistribution (To Avoid Drive Connection Issues)


Meta Denpa for QR
[Production and Personal Inquiries](https://x.com/yu_) /
[Discord community](https://discord.gg/E7KZqxDDP6)
